In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

nguyenng11_spacenewdata_path = kagglehub.dataset_download('nguyenng11/spacenewdata')
nguyenng11_orgdataset_path = kagglehub.dataset_download('nguyenng11/orgdataset')

print('Data source import complete.')


## Setting up...

In [ ]:
import os

# 1. Cài đặt phiên bản mới nhất
!pip install -q -U bitsandbytes>=0.46.1 transformers accelerate


In [1]:
import bitsandbytes
print(f"Phiên bản bitsandbytes hiện tại: {bitsandbytes.__version__}")
# Kết quả phải từ 0.46.1 trở lên mới đạt yêu cầu của Gemma 3.

Phiên bản bitsandbytes hiện tại: 0.49.2


In [4]:
import os
import glob
import json
import torch
import time
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient
import kagglehub

# Lấy Hugging Face token
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("✅ Đã tìm thấy HF_TOKEN.")
except:
    print("❌ Lỗi: Bạn chưa tạo Secret 'HF_TOKEN' trên Kaggle.")

✅ Đã tìm thấy HF_TOKEN.


In [8]:
print("Đang nạp mô hình Gemma 3 12B... Vui lòng đợi trong giây lát.")

# Cấu hình nén 4-bit chuẩn NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Sử dụng bản 12B để đảm bảo tốc độ và sự ổn định trên Kaggle
# model_id = "google/gemma-3-12b-it"
model_path = kagglehub.model_download("google/gemma-3/Transformers/gemma-3-12b-it/1")

tokenizer = AutoTokenizer.from_pretrained(model_path, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("🚀 Mô hình đã sẵn sàng trên GPU!")

Đang nạp mô hình Gemma 3 12B... Vui lòng đợi trong giây lát.


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

🚀 Mô hình đã sẵn sàng trên GPU!


# CONFIG 1 - Zero-Prompt (Không reasoning, chỉ JSON trực tiếp)

In [10]:
import os
import glob
import json
import re
import gc
import torch

# 1. Đường dẫn đến dữ liệu C1 vừa tiền xử lý
TEST_C1_DIR = "/kaggle/input/datasets/nguyenng11/spacenewdata/newdata/test/test.config1"
OUTPUT_PATH = "/kaggle/working/results_config1.jsonl"

# ========================================================
# ZERO-PROMPT: Không có reasoning, không có ví dụ
# Chỉ đưa ra instruction ngắn gọn và yêu cầu JSON output
# ========================================================

# Prompt Giai đoạn 1: Tìm thực thể (ZERO-PROMPT)
PROMPT_ENTITIES_C1 = """Extract all spatial entities from the text below.

Entity types: PLACE, PATH, MOTION.
For each entity, provide: id (e0, e1, ...), text, label, and attributes (form: NAM/NOM, dimensionality: POINT/LINE/AREA, ctv: TRUE/FALSE).

Return ONLY valid JSON in this exact format:
{"entities": [{"id": "e0", "text": "...", "label": "PLACE", "attributes": {"form": "NOM", "dimensionality": "AREA", "ctv": "FALSE"}}]}

TEXT: {text_input}
"""

# Prompt Giai đoạn 2: Tìm quan hệ (ZERO-PROMPT)
PROMPT_RELATIONS_C1 = """Given the text and entities below, extract all spatial relations.

Relation types:
- QSLink: relType can be IN, EC, DC, PO.
- OLink: frame_type can be ABSOLUTE, INTRINSIC, RELATIVE.
- MoveLink: connects MOTION trigger to mover, source, goal.

Use ONLY the provided entity IDs.

Return ONLY valid JSON in this exact format:
{"relations": [{"type": "QSLink", "trajector": "e0", "landmark": "e1", "relType": "EC"}]}

TEXT: {text_input}
ENTITIES: {entities_json}
"""

In [11]:
def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

def extract_json_only(text):
    """Chỉ lấy phần nằm trong dấu { }"""
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {}
    except:
        return {}

def ask_gemma_raw(prompt_text):
    """Zero-prompt: Gửi prompt trực tiếp, không thêm system message phức tạp"""
    messages = [{"role": "user", "content": prompt_text}]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_formatted, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1500, temperature=0.1)

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [12]:
def run_c1_inference(file_path):
    # Đọc văn bản từ file đã tiền xử lý
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'xml')
        raw_text = soup.find('TEXT').get_text()[:2500] # Giới hạn context

    # BƯỚC 1: Tìm thực thể (Zero-Prompt - không reasoning)
    p1 = PROMPT_ENTITIES_C1.replace("{text_input}", raw_text)
    res1 = ask_gemma_raw(p1)
    entities_data = extract_json_only(res1).get("entities", [])

    # BƯỚC 2: Tìm quan hệ (Zero-Prompt - không reasoning)
    entities_str = json.dumps(entities_data, ensure_ascii=False)
    p2 = PROMPT_RELATIONS_C1.replace("{text_input}", raw_text).replace("{entities_json}", entities_str)
    res2 = ask_gemma_raw(p2)
    relations_data = extract_json_only(res2).get("relations", [])

    return {
        "entities": entities_data,
        "relations": relations_data
    }

In [13]:
# Lấy danh sách file từ thư mục đã gọt giũa
all_c1_files = glob.glob(os.path.join(TEST_C1_DIR, '**', '*.xml'), recursive=True)

print(f"🚀 Đang chạy Zero-Prompt Config 1 cho {len(all_c1_files)} file...")

# Xóa file cũ nếu có để chạy lại từ đầu
if os.path.exists(OUTPUT_PATH): os.remove(OUTPUT_PATH)

for i, path in enumerate(all_c1_files):
    name = os.path.basename(path)
    print(f"[{i+1}/{len(all_c1_files)}] Processing: {name}")

    try:
        # Thực hiện suy luận Zero-Prompt
        final_data = run_c1_inference(path)

        # Ghi kết quả sạch vào JSONL
        record = {
            "metadata": {"file": name, "config": "C1"},
            "data": final_data
        }

        with open(OUTPUT_PATH, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    except Exception as e:
        print(f"⚠️ Lỗi tại file {name}: {e}")

    clean_memory()

print(f"🏁 Xong! Kết quả tại: {OUTPUT_PATH}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🚀 Đang chạy Zero-Prompt Config 1 cho 16 file...
[1/16] Processing: Mazatlan.xml
[2/16] Processing: IntoTheAndes.xml
[3/16] Processing: Huaraz.xml


KeyboardInterrupt: 

### Evaluation C1

In [ ]:
import os
import glob
from bs4 import BeautifulSoup

# ĐƯỜNG DẪN ĐẾN THƯ MỤC CHỨA FILE TEST CÒN NGUYÊN TAGS
GOLD_TEST_DIR = "/kaggle/input/datasets/nguyenng11/orgdataset/newdata/test"

def load_gold_standard(directory):
    gold_dict = {}
    files = glob.glob(os.path.join(directory, '**', '*.xml'), recursive=True)

    for path in files:
        fname = os.path.basename(path)
        with open(path, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'xml')
            tags = soup.find('TAGS')
            if tags:
                gold_dict[fname] = tags
    return gold_dict

# Nạp đáp án vào RAM
GOLD_DICT = load_gold_standard(GOLD_TEST_DIR)
print(f"✅ Đã nạp đáp án chuẩn cho {len(GOLD_DICT)} file.")

In [ ]:
import json

def calculate_metrics_fixed(predictions_path, gold_dict):
    stats = {
        "tp": 0, "fp": 0, "fn": 0,
        "correct_attrs": 0, "total_gold_attrs": 0
    }

    link_labels = ['QSLINK', 'OLINK', 'MOVELINK', 'METALINK']

    with open(predictions_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            filename = record['metadata']['file']
            pred_entities = record['data'].get('entities', [])

            if filename not in gold_dict: continue

            # 1. Lấy thực thể Gold (BỎ QUA các thực thể có start = -1)
            gold_tags = gold_dict[filename]
            gold_ents = [
                t for t in gold_tags.find_all(recursive=False)
                if t.name.upper() not in link_labels and t.get('start') != "-1"
            ]

            matched_gold_indices = set()

            # 2. So khớp dựa trên TEXT và LABEL
            for p in pred_entities:
                p_text = str(p.get('text', '')).strip().lower()
                p_label = str(p.get('label', '')).strip().upper()

                is_tp = False
                for idx, g in enumerate(gold_ents):
                    if idx in matched_gold_indices: continue

                    g_text = str(g.get('text', '')).strip().lower()
                    g_label = g.name.upper()

                    if p_text == g_text and p_label == g_label:
                        stats["tp"] += 1
                        matched_gold_indices.add(idx)
                        is_tp = True

                        # Tính Accuracy cho thuộc tính (1c)
                        p_attrs = p.get('attributes', {})
                        for attr, g_val in g.attrs.items():
                            if attr in ['id', 'start', 'end', 'text', 'comment']: continue
                            stats["total_gold_attrs"] += 1
                            if str(p_attrs.get(attr, '')).upper() == str(g_val).upper():
                                stats["correct_attrs"] += 1
                        break

                if not is_tp:
                    stats["fp"] += 1

            # 3. Tính FN
            stats["fn"] += (len(gold_ents) - len(matched_gold_indices))

    # TÍNH TOÁN THEO CÔNG THỨC TÀI LIỆU
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0
    acc = stats["correct_attrs"] / stats["total_gold_attrs"] if stats["total_gold_attrs"] > 0 else 0

    return {"Precision": p, "Recall": r, "F1": f1, "Accuracy": acc}

# Chạy lại chấm điểm
results = calculate_metrics_fixed(OUTPUT_PATH, GOLD_DICT)
print("\n--- KẾT QUẢ ĐÁNH GIÁ CONFIG 1 (ZERO-PROMPT) ---")
for k, v in results.items():
    print(f"📌 {k}: {v:.4f}")

In [ ]:
import json
from collections import defaultdict

def evaluate_5_tasks(predictions_path, gold_dict):
    # Khởi tạo bộ đếm cho 5 bài toán
    results = {
        "1a_1b": {"tp": 0, "fp": 0, "fn": 0}, # Entity Type
        "1c": {"correct": 0, "total": 0},      # Entity Attributes
        "1d": {"tp": 0, "fp": 0, "fn": 0},     # Link Type
        "1e": {"correct": 0, "total": 0}       # Link Attributes
    }

    link_labels = ['QSLINK', 'OLINK', 'MOVELINK']

    with open(predictions_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            filename = record['metadata']['file']
            pred_data = record['data']
            if filename not in gold_dict: continue

            gold_tags = gold_dict[filename]
            all_gold = gold_tags.find_all(recursive=False)
            gold_ents = [t for t in all_gold if t.name.upper() not in link_labels and t.get('start') != "-1"]
            gold_links = [t for t in all_gold if t.name.upper() in link_labels]

            # --- CHẤM BÀI 1a, 1b, 1c (ENTITIES) ---
            pred_ents = pred_data.get('entities', [])
            matched_gold_ids = {}

            for p in pred_ents:
                p_text = str(p.get('text', '')).strip().lower()
                p_label = str(p.get('label', '')).strip().upper()

                match = None
                for g in gold_ents:
                    if p_text == str(g.get('text', '')).strip().lower() and p_label == g.name.upper():
                        match = g
                        break

                if match:
                    results["1a_1b"]["tp"] += 1
                    matched_gold_ids[p.get('id')] = match.get('id')

                    # Bài 1c: Attributes
                    p_attrs = p.get('attributes', {})
                    for attr, g_val in match.attrs.items():
                        if attr in ['id', 'start', 'end', 'text', 'comment']: continue
                        results["1c"]["total"] += 1
                        if str(p_attrs.get(attr, '')).upper() == str(g_val).upper():
                            results["1c"]["correct"] += 1
                else:
                    results["1a_1b"]["fp"] += 1

            results["1a_1b"]["fn"] += (len(gold_ents) - len(matched_gold_ids))

            # --- CHẤM BÀI 1d, 1e (RELATIONS) ---
            pred_rels = pred_data.get('relations', [])
            matched_link_count = 0

            for pr in pred_rels:
                p_type = pr.get('type', '').upper()
                is_link_tp = False

                for gr in gold_links:
                    if p_type == gr.name.upper():
                        results["1d"]["tp"] += 1
                        is_link_tp = True
                        matched_link_count += 1

                        # Bài 1e: Link Attributes
                        p_link_attrs = pr.get('attributes', {})
                        for g_attr, g_val in gr.attrs.items():
                            if g_attr in ['id', 'trajector', 'landmark', 'fromID', 'toID', 'comment']: continue
                            results["1e"]["total"] += 1
                            if str(pr.get(g_attr, '')).upper() == str(g_val).upper():
                                results["1e"]["correct"] += 1
                        break

                if not is_link_tp:
                    results["1d"]["fp"] += 1

            results["1d"]["fn"] += (len(gold_links) - matched_link_count)

    return results

def display_report(res):
    print(f"{'BÀI TOÁN (TASK)':<25} | {'P':<8} | {'R':<8} | {'F1':<8} | {'ACC':<8}")
    print("-" * 70)

    # Tính 1a_1b
    tp, fp, fn = res["1a_1b"]["tp"], res["1a_1b"]["fp"], res["1a_1b"]["fn"]
    p = tp/(tp+fp) if tp+fp > 0 else 0
    r = tp/(tp+fn) if tp+fn > 0 else 0
    f1 = 2*p*r/(p+r) if p+r > 0 else 0
    print(f"{'1a+1b: Entity Label':<25} | {p:.4f} | {r:.4f} | {f1:.4f} | {'-':<8}")

    # Tính 1c
    acc_1c = res["1c"]["correct"]/res["1c"]["total"] if res["1c"]["total"] > 0 else 0
    print(f"{'1c: Entity Attributes':<25} | {'-':<8} | {'-':<8} | {'-':<8} | {acc_1c:.4f}")

    # Tính 1d
    tp_d, fp_d, fn_d = res["1d"]["tp"], res["1d"]["fp"], res["1d"]["fn"]
    p_d = tp_d/(tp_d+fp_d) if tp_d+fp_d > 0 else 0
    r_d = tp_d/(tp_d+fn_d) if tp_d+fn_d > 0 else 0
    f1_d = 2*p_d*r_d/(p_d+r_d) if p_d+r_d > 0 else 0
    print(f"{'1d: Relation Type':<25} | {p_d:.4f} | {r_d:.4f} | {f1_d:.4f} | {'-':<8}")

    # Tính 1e
    acc_1e = res["1e"]["correct"]/res["1e"]["total"] if res["1e"]["total"] > 0 else 0
    print(f"{'1e: Relation Attributes':<25} | {'-':<8} | {'-':<8} | {'-':<8} | {acc_1e:.4f}")

# Thực thi
final_metrics = evaluate_5_tasks(OUTPUT_PATH, GOLD_DICT)
display_report(final_metrics)